In [1]:
# KaiWang V-0417-01
%load_ext autoreload
%autoreload 2
import pandas as pd 
import numpy as np
from chinese_calendar import is_holiday, is_workday, is_in_lieu
from chinese_calendar import get_holidays
from dotenv import load_dotenv
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict
import time 
import json
import os 
import re
import warnings
warnings.filterwarnings("ignore")

import HelperFunc as hfc
from Prompt_Template import PromptTemplate

# 企业实践课程安排第二周课题
结合第一周对K7直播间业务的理解，思考企业可能依据哪些核心指标（如GMV、互动率等）进行的初步分类。

# 任务一：多源数据清洗、对齐与整合
1. 视频文件与分类清单对齐：确认提供的视频文件与分类清单完全匹配，处理任何不一致的情况。
2. 数据清洗与合并：获取与所有视频对应的每日绩效数据，清洗异常值与缺失值。将清洗后的绩效数据，与已有的“爆款/非爆款”分类标签进行精确关联与合并。
3. 业务数据整合：尝试整合视频的文本信息（标题、描述）、发布时段等其他可用属性。

## Kai 数据产出说明

### 1) dy_daily_perf_with_mv_match.xlsx
- 内容：金运抖音每日绩效主表 + 视频文件匹配结果（基于标准化标题/文件名匹配）。
- 作用：用于分析“每日投放表现”与“视频是否在爆款/普通素材目录中”的关系，可直接做分组统计和可视化。

### 2) dy_meaningful_titles_analysis_result.xlsx
- 内容：金运有意义标题的 LLM 评分结果（marketing/attention/promotion）+ embedding 向量信息。
- 作用：用于标题质量评估、语义相似度分析、向量特征建模（如聚类、检索、下游预测特征）。

### 3) k7_material_desc_with_mv_match.xlsx
- 内容：K7 素材报表清洗后的主表 + 视频匹配结果（是否匹配到爆款/普通目录）。
- 作用：用于分析 K7 素材侧的结构化指标（点击、转化、消耗等）与素材分类标签之间的关系。

### 4) k7_meaningful_titles_analysis_result.xlsx
- 内容：K7 有意义标题的 LLM 评分结果 + embedding 向量信息。
- 作用：用于 K7 素材标题语义分析、相似标题检索，以及与绩效指标做联合分析。

### 5) kinno_dy_daily_perf_processed.xlsx
- 内容：金运抖音每日绩效数据的清洗后基础表（含时间特征、节假日特征、标题规则特征等）。
- 作用：作为后续所有分析的基础底表，支持与 LLM/embedding 结果、视频匹配结果进行主键或标题维度合并。

## 使用建议
- 如果做业务复盘：优先看 dy_daily_perf_with_mv_match.xlsx 与 k7_material_desc_with_mv_match.xlsx。
- 如果做语义或算法特征：优先看 dy_meaningful_titles_analysis_result.xlsx 与 k7_meaningful_titles_analysis_result.xlsx。
- 如果做统一数据建模：以 kinno_dy_daily_perf_processed.xlsx 为基础再逐步 merge 其他结果表。


In [2]:
ANALYSIS_DIR = r"..//Analysis"
DATA_DIR = r"..//Data"
Processed_DATA_DIR = r"..//Processed_Data"
os.makedirs(ANALYSIS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(Processed_DATA_DIR, exist_ok=True)

# Folder
KINNO_DATA_FOLDER = os.path.join(DATA_DIR, "金运") # 金运数据文件夹
KINNO_DY_DAILY_PERF_DATA_FOLDER = os.path.join(KINNO_DATA_FOLDER, "抖音每日绩效数据") # 金运抖音数据文件夹
OCEANENGINE_DATA_FOLDER = os.path.join(DATA_DIR, "巨量千川") # 巨量千川数据文件夹
TECDO_DATA_FOLDER = os.path.join(DATA_DIR, "钛动科技") # 钛动科技数据文件夹
TECDO_GOOD_PERF_DATA_FOLDER = os.path.join(TECDO_DATA_FOLDER, "绩效表现好的视频文件") # 钛动科技绩效表现好的视频文件夹
TECDO_BAD_PERF_DATA_FOLDER = os.path.join(TECDO_DATA_FOLDER, "绩效表现差的视频文件") # 钛动科技绩效表现差的视频文件夹
K7_DATA_FOLDER = os.path.join(DATA_DIR, "K7视频素材") # K7视频素材文件夹
K7_GOOD_PERF_DATA_FOLDER = os.path.join(K7_DATA_FOLDER, "爆款") # K7绩效表现好的视频文件夹
K7_BAD_PERF_DATA_FOLDER = os.path.join(K7_DATA_FOLDER, "没啥动静") # K7绩效表现差的视频文件夹

ENV_FILE = r"..//.env" # 环境变量文件路径
load_dotenv(ENV_FILE)

True

In [3]:
# 中国假日数据获取
# 获取 2022 到 2026 的数据
all_holidays_df = pd.DataFrame()
for year in range(2022, 2027):
    df_year = hfc.fetch_holiday_details(year)
    all_holidays_df = pd.concat([all_holidays_df, df_year], ignore_index=True)
all_holidays_df.rename(columns={'name':'holiday_name','date': 'datestamp'}, inplace=True) # 将date列重命名为datestamp以便与金运数据合并
all_holidays_df['datestamp'] = pd.to_datetime(all_holidays_df['datestamp']) # 将datestamp列转换为datetime格式
all_holidays_df.to_csv(os.path.join(Processed_DATA_DIR, "chinese_holidays_2022_2026.csv"), index=False)
all_holidays_df = pd.read_csv(os.path.join(Processed_DATA_DIR, "chinese_holidays_2022_2026.csv"),parse_dates=['datestamp']) # isOffDay 为 True 代表放假, False 代表工作日, 还有一些特殊情况如调休、补班等
display(all_holidays_df)

,holiday_name,datestamp,isOffDay
0,元旦,2022-01-01,True
1,元旦,2022-01-02,True
2,元旦,2022-01-03,True
3,春节,2022-01-29,False
4,春节,2022-01-30,False
...,...,...,...
175,国庆节,2026-10-04,True
176,国庆节,2026-10-05,True
177,国庆节,2026-10-06,True
178,国庆节,2026-10-07,True


# 1. 金运 - 抖音每日绩效数据

## 一、文件合并与时间戳提取
| 操作 | 说明 |
|------|------|
| 文件读取 | 遍历每日 xlsx 文件，跳过临时文件（~$ 开头） |
| 时间戳提取 | 用正则从文件名提取时间范围，取起始时间生成 datestamp |
| DataFrame 合并 | 使用 pd.concat 合并所有日数据 |

## 二、基础清洗与字段拆解
| 字段/操作 | 处理方式 |
|------|------|
| 素材ID、素材创建时间、datestamp | 作为基础字段保留到处理后主表 |
| 素材标题 | 拆分为“素材名称”和“素材标签”两列 |
| 素材名称 | 去除 #标签 后保留纯文本 |
| 素材标签 | 提取所有 #标签；空列表转为 NaN |
| 素材时长 | 在 MM:SS 前补 00:，转 timedelta 后取秒数 |

## 三、类型转换与数值标准化
| 类型 | 字段 | 处理方式 |
|------|------|----------|
| 类别编码 | 素材类型、素材来源 | astype('category') 后使用 cat.codes 编码，并输出映射关系 |
| 百分比列 | 点击率、转化率、2秒播放率、3秒播放率、5秒播放率、10秒播放率、完播率 | 去掉 % 后转 float |
| 普通数值列 | 消耗、展示次数、点击次数等 | 先去逗号，再转 float |
| 日期列 | datestamp | 统一转 datetime |

## 四、节假日与时间特征扩展
| 新增特征 | 含义 |
|------|------|
| holiday_name | 与中国节假日表按 datestamp 左连接，缺失填 workday |
| isOffDay | 是否休息日，缺失填 False |
| weekday | 星期几（0-6，0=周一） |
| 被调休的周末 | 是否属于调休补班周末 |
| 休息的周中 | 是否属于工作周中的休息日 |
| month | 月份 |
| shopping_festival | 是否命中双11/双12 |

## 五、标题可读性与语义规则特征
| 新增特征 | 处理逻辑 |
|------|----------|
| meaningful_title | 用正则识别 gibberish（纯数字、日期、混剪、mp4 文件名等），并取反得到“有意义标题”标记 |
| date_gibberish_title_ratio | 按天统计 gibberish 标题占比 |
| rollong_5d_giibbersh_ratio | gibberish 占比 5 日滚动均值 |
| family_kw_in_title | 标题是否包含家庭关系关键词 |
| device_kw_in_title | 标题是否包含设备相关关键词 |
| singing_kw_in_title | 标题是否包含唱歌相关关键词 |
| negation_kw_in_title | 标题是否包含否定/劝退类词汇 |
| beauty_kw_in_title | 标题是否包含美女/网红等词汇 |

## 六、字段组织与输出说明
- 列分组字典：kinno_dy_daily_perf_df_cols_dict
  - numeric_cols：普通数值列（含素材时长）
  - pct_cols：百分比列
  - normal_cols：基础字段与标题拆解字段
- 当前单元最终产物为内存中的 kinno_dy_daily_perf_df_processed，供后续 LLM 标题分析与主表合并步骤继续使用。
- 如需落盘，可使用下方已预留的 to_excel 语句（当前代码中为注释状态）。

In [ ]:
KINNO_DY_DAILY_PERF_FILES = os.listdir(KINNO_DY_DAILY_PERF_DATA_FOLDER) # 金运数据文件列表
timerange_pattern = r'(\d{4}-\d{2}-\d{2} \d{2}_\d{2}_\d{2})'
kinno_dy_daily_perf_dfs = [] # 金运数据DataFrame列表
for kinno_dy_daily_perf_file in tqdm(KINNO_DY_DAILY_PERF_FILES, desc = 'Processing KINNO DY Daily Performance Files'):
    if kinno_dy_daily_perf_file.startswith("~$"): continue # 跳过临时文件
    timerange_match = re.findall(timerange_pattern, kinno_dy_daily_perf_file)
    start_time_str, end_time_str = timerange_match[0], timerange_match[1] if len(timerange_match) > 1 else None
    date_stamp = pd.to_datetime(start_time_str, format = "%Y-%m-%d %H_%M_%S").date() if start_time_str else None # 将开始时间字符串转换为日期对象
    df = pd.read_excel(os.path.join(KINNO_DY_DAILY_PERF_DATA_FOLDER, kinno_dy_daily_perf_file))
    #############################################
    df['datestamp'] = date_stamp # 将日期对象添加为新列
    #############################################
    kinno_dy_daily_perf_dfs.append(df)
print(f"金运数据文件数量: {len(KINNO_DY_DAILY_PERF_FILES)}\n")
kinno_dy_daily_perf_df = pd.concat(kinno_dy_daily_perf_dfs, ignore_index = True) # 合并所有DataFrame
kinno_dy_daily_perf_df_columns = kinno_dy_daily_perf_df.columns.tolist() # 金运数据DataFrame的列名列表
kinno_dy_daily_perf_df_processed = kinno_dy_daily_perf_df[['素材ID','素材创建时间','datestamp']].copy() # 金运数据处理后的DataFrame副本
# kinno_dy_daily_perf_df.info() # Brief info
print('缺失值数量(ascending): ')
display(kinno_dy_daily_perf_df.replace("-", np.nan).isna().sum().sort_values(ascending=False).head())

#############################################
# 处理数据
#############################################
remove_comma_func = lambda x: str(x).replace(',', '') if isinstance(x, str) else x # function: 移除字符串中的逗号
# 素材标题 转化为 素材名称 + 素材标签(tags)
kinno_dy_daily_perf_df['素材标题'] = kinno_dy_daily_perf_df['素材标题'].astype(str)
tags_pattern = r'#([^#]+)' # 匹配 #标签
kinno_dy_daily_perf_df_processed['素材名称'] = kinno_dy_daily_perf_df['素材标题'].str.replace(tags_pattern, '', regex=True) # 提取标签列表
kinno_dy_daily_perf_df_processed['素材标签'] = kinno_dy_daily_perf_df['素材标题'].str.findall(tags_pattern) # 去除标签后的文字
kinno_dy_daily_perf_df_processed['素材标签'] = kinno_dy_daily_perf_df_processed['素材标签'].apply(lambda x: x if len(x) > 0 else np.nan)
print("素材标签有些行是空列表, 已经替换为NaN")
# 素材时长 统一转化为秒
time_fixed = '00:' + kinno_dy_daily_perf_df['素材时长'].astype(str)
kinno_dy_daily_perf_df_processed['素材时长'] = pd.to_timedelta(time_fixed, errors='coerce').dt.total_seconds()
# 素材类型 转为category
kinno_dy_daily_perf_df_processed['素材类型'] = kinno_dy_daily_perf_df['素材类型'].astype('category')
material_type_mapping = {cat: idx for idx, cat in enumerate(kinno_dy_daily_perf_df_processed['素材类型'].cat.categories)}
kinno_dy_daily_perf_df_processed['素材类型'] = kinno_dy_daily_perf_df_processed['素材类型'].cat.codes # 将分类变量转换为数值编码
# 素材来源 转为category ('-' 代表没有来源) 
kinno_dy_daily_perf_df_processed['素材来源'] = kinno_dy_daily_perf_df['素材来源'].astype('category')
material_source_mapping = {cat: idx for idx, cat in enumerate(kinno_dy_daily_perf_df_processed['素材来源'].cat.categories)}
kinno_dy_daily_perf_df_processed['素材来源'] = kinno_dy_daily_perf_df_processed['素材来源'].cat.codes # 将分类变量转换为数值编码
print(f"素材类型映射, 变量名: material_type_mapping : {material_type_mapping}")
print(f"素材来源映射, 变量名: material_source_mapping : {material_source_mapping}")
# 其他列
kinno_dy_daily_perf_other_cols = ['素材ID','素材创建时间','datestamp','素材名称','素材标签','素材类型','素材来源']
# 百分比列转为数值类型 (单位都是%)
kinno_dy_daily_perf_percentage_cols = ['点击率','转化率','2秒播放率','3秒播放率','5秒播放率','10秒播放率','完播率']
for col in kinno_dy_daily_perf_percentage_cols:
    kinno_dy_daily_perf_df_processed[col] = kinno_dy_daily_perf_df[col].apply(lambda x: float(str(x).replace('%', '')) if isinstance(x, str) else np.nan)
# 普通数值列转为数值类型
kinno_dy_daily_perf_normal_numeric_cols = [c for c in kinno_dy_daily_perf_df_columns if c not in kinno_dy_daily_perf_other_cols + kinno_dy_daily_perf_percentage_cols + ['素材标题','素材时长','标签']]
for col in kinno_dy_daily_perf_normal_numeric_cols:
    kinno_dy_daily_perf_df_processed[col] = kinno_dy_daily_perf_df[col].apply(remove_comma_func).astype(float)
kinno_dy_daily_perf_normal_numeric_cols = kinno_dy_daily_perf_normal_numeric_cols + ['素材时长'] # 将素材时长也加入普通数值列列表
kinno_dy_daily_perf_df_cols_dict = {'numeric_cols': kinno_dy_daily_perf_normal_numeric_cols, 'pct_cols': kinno_dy_daily_perf_percentage_cols, 'normal_cols': kinno_dy_daily_perf_other_cols}
kinno_dy_daily_perf_df_processed['datestamp'] = pd.to_datetime(kinno_dy_daily_perf_df_processed['datestamp']) # 将datestamp列转换为datetime格式

################################################################
# 老板说影响因素有很多: 时间，标题，人物颜值(这个要ai打分吧)等
################################################################
# 添加节假日信息
holiday_names = all_holidays_df['holiday_name'].unique().tolist()
print(f"节假日名称列表: {holiday_names}")
kinno_dy_daily_perf_df_processed = pd.merge(kinno_dy_daily_perf_df_processed, all_holidays_df, on = 'datestamp', how = 'left') # 将节假日信息合并到金运数据中
kinno_dy_daily_perf_df_processed['holiday_name'] = kinno_dy_daily_perf_df_processed['holiday_name'].fillna('workday') # 将没有节假日名称的行填充为'workday'
kinno_dy_daily_perf_df_processed['isOffDay'] = kinno_dy_daily_perf_df_processed['isOffDay'].fillna(False) # 将没有isOffDay信息的行填充为False (工作日)
kinno_dy_daily_perf_df_processed['weekday'] = kinno_dy_daily_perf_df_processed['datestamp'].apply(lambda x: x.weekday() if pd.notna(x) else np.nan) # datestamp列的星期几 (0-6, 0代表周一)
kinno_dy_daily_perf_df_processed['被调休的周末'] = kinno_dy_daily_perf_df_processed['datestamp'].apply(lambda x: is_in_lieu(x) if pd.notna(x) else np.nan) # 判断是否为被调休的周末
kinno_dy_daily_perf_df_processed['休息的周中'] = kinno_dy_daily_perf_df_processed['datestamp'].apply(lambda x: is_holiday(x) and not is_workday(x) if pd.notna(x) else np.nan) # 判断是否为休息的周中
kinno_dy_daily_perf_df_processed['month'] = kinno_dy_daily_perf_df_processed['datestamp'].dt.month # datestamp列的月份
kinno_dy_daily_perf_df_processed['shopping_festival'] = kinno_dy_daily_perf_df_processed['datestamp'].apply(lambda x: (x.month == 11 and x.day == 11) or (x.month == 12 and x.day == 12) if pd.notna(x) else False) # 判断是否为双11或双12

#############################################################################################################################
# 标题的作用是给抖音的算法来看，所以(1)Gibberish标题可能会影响观感和算法推送效果 (2) 触发关键字，推送de人群不一样(我猜的)
#############################################################################################################################
# 素材名称是否是meaningful的, 很多标题都是gibberish, 例如纯数字、日期、混剪、或者直接就是视频文件名等
gibberish_title_pattern = re.compile(r'^\d+$|\d+月\d+日|混剪|\.mp4$|\(\d+\)')
gibberish_title_filter = lambda x: bool(gibberish_title_pattern.search(x))
kinno_dy_daily_perf_df_processed['meaningful_title'] = kinno_dy_daily_perf_df_processed['素材名称'].apply(gibberish_title_filter) ^ 1

# 当日/近5日gibberish标题比例 (我觉得这个很影响观感....)
daily_gibberish_title_ratio = kinno_dy_daily_perf_df_processed.groupby('datestamp')['meaningful_title'].apply(lambda x: 1 - x.mean()).reset_index(name='date_gibberish_title_ratio')
daily_gibberish_title_ratio['rollong_5d_giibbersh_ratio'] = daily_gibberish_title_ratio['date_gibberish_title_ratio'].rolling(window=5, min_periods=1).mean() # 计算当日gibberish标题比例的5日滚动平均
kinno_dy_daily_perf_df_processed = pd.merge(kinno_dy_daily_perf_df_processed, daily_gibberish_title_ratio, on = 'datestamp', how = 'left')

# 含人的标题
family_title_keywords = ['爸爸', '妈妈', '儿子', '女儿', '兄弟', '姐妹', '爷爷', '奶奶', '外公', '外婆','老公','老婆','丈夫','妻子','姐','妹','哥','弟','父亲','母亲','儿女','兄妹','祖父','祖母','外祖父','外祖母','小伙子','家庭','家']
kinno_dy_daily_perf_df_processed['family_kw_in_title'] = kinno_dy_daily_perf_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in family_title_keywords))

# 设备标题
device_title_keywords = ['设施','设备','麦克风','音响','k歌','K歌','蓝牙','KTV']
kinno_dy_daily_perf_df_processed['device_kw_in_title'] = kinno_dy_daily_perf_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in device_title_keywords))

# 唱歌标题
singing_title_keywords = ['唱歌','唱了','唱了首','唱了个','唱了段','唱了首歌','唱了个歌','唱了段歌','唱了首曲','唱了个曲','唱了段曲','唱了首调','麦霸']
kinno_dy_daily_perf_df_processed['singing_kw_in_title'] = kinno_dy_daily_perf_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in singing_title_keywords))

# 否定词 (我发现有些标题里有否定词.....无语了)
negation_title_keywords = ['不要','千万别','不能','不行','别再','别看','别点','别点开','别点进去','别点了','别点那个','别点这个','别点那个了','别点这个了']
kinno_dy_daily_perf_df_processed['negation_kw_in_title'] = kinno_dy_daily_perf_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in negation_title_keywords))

# 美女, 网红...... 
beauty_title_keywords = ['美女', '网红', '女神', '小姐姐', '小仙女']
kinno_dy_daily_perf_df_processed['beauty_kw_in_title'] = kinno_dy_daily_perf_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in beauty_title_keywords))

kinno_dy_daily_perf_df_processed['normalized_title'] = kinno_dy_daily_perf_df_processed['素材名称'].apply(hfc.normalize_name) # 对素材名称进行统一处理, 方便后续匹配视频
print("KINNO抖音每日绩效数据处理完成")


Processing KINNO DY Daily Performance Files: 100%|██████████| 279/279 [00:16<00:00, 17.32it/s]


金运数据文件数量: 279

缺失值数量(ascending): 


标签        12588
素材来源       1007
素材标题          0
素材ID          0
素材创建时间        0
dtype: int64

素材标签有些行是空列表, 已经替换为NaN
素材类型映射, 变量名: material_type_mapping : {'竖版视频': 0}
素材来源映射, 变量名: material_source_mapping : {'-': 0, '抖音号主页素材': 1, '本地上传': 2}
节假日名称列表: ['元旦', '春节', '清明节', '劳动节', '端午节', '中秋节', '国庆节', '中秋节、国庆节', '国庆节、中秋节']
KINNO抖音每日绩效数据处理完成


# 2. 用LLM (我用的是GLM-4.7) parse 素材名称 + Embedding 表征
## 目的
- 对素材名称进行语义营销能力评估，补充规则特征难以覆盖的表达质量信息。
- 在前面已剔除 gibberish 标题（meaningful_title == True）的基础上，仅分析有意义标题，减少无效调用与噪声。
- 额外为每个标题生成 embedding 向量，用于后续相似素材检索、聚类分析和向量特征建模。

## 输入与预处理
- 输入数据来自 kinno_dy_daily_perf_df_processed 中的素材名称。
- 先去重得到 dy_meaningful_titles，再转为 DataFrame 做快速抽样检查。

## 评估流程（LLM 打分）
- 调用 hfc.evaluate_titles_multithread(...) 进行并发评估。
- 使用环境变量 SILICONFLOW_API 作为 API Key。
- 提示词模板来自 PromptTemplate.TITLE_EVAL_PROMPT。
- 并发线程数设置为 MAX_WORKERS（代码里当前为 16）。
- 每条返回结果包含：
  - marketing_score：营销表达得分
  - attention_score：吸引力得分
  - promotion_score：促转化得分
  - status：调用状态（success / failed）
  - error：失败时的错误信息

## Embedding 流程（语义向量）
- 调用 hfc.embed_titles_multithread(...) 对同一批 dy_meaningful_titles 生成向量表示。
- 当前 embedding 模型为 BAAI/bge-large-zh-v1.5。
- 每条返回结果包含：
  - embedding：标题向量
  - embedding_dim：向量维度
  - model：模型名称
  - status / error：调用状态与错误信息
- 将 embedding 结果与 LLM 打分结果按 meaningful_title 左连接，形成统一结果表（suffix: _eval / _embed）。

## 结果清洗与统计
- 将三项分数字段统一转为数值类型（无法转换记为 NaN）。
- 输出总耗时、成功条数、失败条数。
- 预览前几行结果，检查字段完整性。

## 输出产物
- 结果表保存为：Processed_Data/dy_meaningful_titles_analysis_result.xlsx。
- 结果表同时包含打分特征与 embedding 特征，可按标题回填到主表，作为建模、分层分析与相似度分析特征。

In [ ]:
# 对标题进行分析, 剔除gibberish标题
dy_meaningful_titles = kinno_dy_daily_perf_df_processed[kinno_dy_daily_perf_df_processed['meaningful_title'] == True]['素材名称'].unique()
print(f"有 {len(dy_meaningful_titles)} 个不同的有意义的标题")
dy_meaningful_titles_df = pd.DataFrame(dy_meaningful_titles, columns=['meaningful_title'])
display(dy_meaningful_titles_df.head())

# # 配置
# MAX_RETRIES = 3  # 最大重试次数
# MAX_WORKERS = 16 # 并发线程数
# TIMTOUT = 60 # 每个请求的超时时间（秒）

# # 执行多线程评估
# print(f"开始评估 {len(dy_meaningful_titles)} 个标题，使用 {MAX_WORKERS} 个线程...")
# start_time = time.time()
# # 函数见HelperFunc.py, 提示词见Prompt_Template.py, 评估结果包含: marketing_score, attention_score, promotion_score, status(success/failed), error_message(if failed)等字段
# results = hfc.evaluate_titles_multithread(
#     titles=dy_meaningful_titles,
#     api_key= os.getenv("SILICONFLOW_API"),
#     prompt_template = PromptTemplate.TITLE_EVAL_PROMPT,
#     max_workers=16  # 可根据需要调整
# )
# elapsed_time = time.time() - start_time
# dy_meaningful_titles_analysis_result_df = pd.DataFrame(results)

# # 数值转换
# if not dy_meaningful_titles_analysis_result_df.empty:
#     # 转换分数列为数值类型
#     score_cols = ['marketing_score', 'attention_score', 'promotion_score']
#     for col in score_cols:
#         if col in dy_meaningful_titles_analysis_result_df.columns:
#             dy_meaningful_titles_analysis_result_df[col] = pd.to_numeric(dy_meaningful_titles_analysis_result_df[col], errors='coerce')

# # 输出统计信息
# print(f"\n完成, 耗时: {elapsed_time:.2f} 秒")
# print(f"成功: {dy_meaningful_titles_analysis_result_df['status'].value_counts().get('success', 0)} 条")
# print(f"失败: {dy_meaningful_titles_analysis_result_df['status'].value_counts().get('failed', 0)} 条")
# dy_meaningful_titles_analysis_result_df['source'] = 'dy_daily_perf'

# dy_meaningful_titles_embedding_results = hfc.embed_titles_multithread(
#     titles=dy_meaningful_titles,
#     api_key=os.getenv("SILICONFLOW_API"),
#     model="BAAI/bge-large-zh-v1.5",
#     max_workers=8,
#     timeout=40,
#     max_retries=3,
# )
# dy_meaningful_titles_embedding_df = pd.DataFrame(dy_meaningful_titles_embedding_results)
# dy_meaningful_titles_analysis_result_df = pd.merge(dy_meaningful_titles_analysis_result_df, dy_meaningful_titles_embedding_df, on = 'meaningful_title', how='left',suffixes=('_eval', '_embed'))

# # 将结果保存到Excel文件
# dy_meaningful_titles_analysis_result_df.to_excel(os.path.join(Processed_DATA_DIR, "dy_meaningful_titles_analysis_result.xlsx"), index=False)

# 结果读取和展示
dy_meaningful_titles_analysis_result_df = pd.read_excel(os.path.join(Processed_DATA_DIR, "dy_meaningful_titles_analysis_result.xlsx")) # 读取评估结果
# 显示结果
print("\n结果预览:")
display(dy_meaningful_titles_analysis_result_df.head())


有 221 个不同的有意义的标题


,meaningful_title
0,老婆瞒着我在家唱歌，被震惊了！
1,全新升级的K歌蓝牙音响，搭配两支无线麦克风让你唱歌方便！
2,老婆又瞒着我偷偷在家唱K！
3,没想到美女用这台设备唱歌太好听了
4,公园寻找实力唱将！


开始评估 221 个标题，使用 16 个线程...


评估标题进度: 100%|██████████| 221/221 [09:22<00:00,  2.55s/个, 成功=221, 失败=0] 



完成, 耗时: 562.86 秒
成功: 221 条
失败: 0 条


Embedding进度: 100%|██████████| 221/221 [00:06<00:00, 34.76个/s, 成功=221, 失败=0]



结果预览:


,marketing_score,attention_score,promotion_score,keywords,overall_comment,meaningful_title,status_eval,error_eval,source,status_embed,error_embed,embedding,embedding_dim,model
0,3,5,4,"['老婆', '偷偷', '在家', '唱K']",生活化场景引发共鸣，好奇心强，适合软广植入。,老婆又瞒着我偷偷在家唱K！,success,NaN,dy_daily_perf,success,NaN,"[-0.0049476554, 0.01017018, -0.064548574, -0.0...",1024,BAAI/bge-large-zh-v1.5
1,4,4,4,"['唱歌', '网红', '羡慕', '户外']",场景定位精准，利用网红效应吸引眼球，转化导向强。,让你唱歌连网红都羡慕，户外唱歌就用它！,success,NaN,dy_daily_perf,success,NaN,"[0.02915465, -0.008279354, 0.013504076, -0.016...",1024,BAAI/bge-large-zh-v1.5
2,3,5,4,"['隐蔽', '发现']",好奇心强，吸引点击，但缺乏具体卖点,这么隐蔽还是被我发现了！,success,NaN,dy_daily_perf,success,NaN,"[-0.0009454696, -0.032502543, -0.029514087, -0...",1024,BAAI/bge-large-zh-v1.5
3,3,3,3,"['公园', '寻找', '实力唱将']",标题描述了活动内容，但缺乏强烈的吸引点或利益诱导。,公园寻找实力唱将！,success,NaN,dy_daily_perf,success,NaN,"[0.040860176, -0.017653607, -0.01757709, 0.034...",1024,BAAI/bge-large-zh-v1.5
4,3,4,3,"['美女', '设备', '唱歌', '好听']",美女吸睛，强调音质，但缺乏具体产品名，转化稍弱。,没想到美女用这台设备唱歌太好听了,success,NaN,dy_daily_perf,success,NaN,"[0.041671805, -0.001673365, -0.0109963985, -0....",1024,BAAI/bge-large-zh-v1.5


# 3. K7视频素材
- 爆款 (视频文件, 需要从阿里云盘里面下载, 没上传)
- 没啥动静 (视频文件, 需要从阿里云盘里面下载, 没上传)
- 素材数据报表.xlsx
  - 标签依旧全是`-`, Drop this col

In [6]:
k7_material_desc_df = pd.read_excel(os.path.join(K7_DATA_FOLDER, "素材数据报表.xlsx")).drop(columns = ['标签']) # K7素材数据DataFrame
# k7_material_desc_df.info()
print('缺失值数量(ascending): ')
display(k7_material_desc_df.replace("-", np.nan).isna().sum().sort_values(ascending=False).head(7).to_frame().T)

# 素材ID 为'-'的行
print("素材ID为'-'的行: ")
display(k7_material_desc_df[k7_material_desc_df['素材ID'] == '-'])
print("只有一行, AIGC动态创意素材集合, 这一条并没有出现在上面的KINNO抖音每日绩效数据中, 可能是一个特殊的素材集合, 但由于没有对应的素材ID, 无法与绩效数据进行关联分析")

# 去掉AIGC动态创意素材集合这一行后, 处理数据
k7_material_desc_df = k7_material_desc_df[k7_material_desc_df['素材ID'] != '-'].copy() # 去掉素材ID为'-'的行
print('去除AIGC这一行后 缺失值数量(ascending): ')
display(k7_material_desc_df.replace("-", np.nan).isna().sum().sort_values(ascending=False).head(5).to_frame().T)
k7_material_desc_df['素材ID'] = k7_material_desc_df['素材ID'].astype(int) # 将素材ID列转换为字符串类型

# 同样, 对数据进行清理
k7_material_desc_df_processed = k7_material_desc_df[['素材ID']].copy()
# 素材名称
k7_material_desc_df_processed['素材名称'] = k7_material_desc_df['素材名称'].apply(lambda x: x.split(".mp4")[0] if isinstance(x, str) else x) # 去掉.mp4后缀, 看看有多少不同的素材名称
# 素材评估
k7_material_desc_df_processed['素材评估'] = k7_material_desc_df['素材评估'].astype('category')
material_eval_mapping = {cat: idx for idx, cat in enumerate(k7_material_desc_df_processed['素材评估'].cat.categories)}
k7_material_desc_df_processed['素材评估'] = k7_material_desc_df_processed['素材评估'].cat.codes # 将分类变量转换为数值编码
# 素材时长
time_fixed = '00:' + k7_material_desc_df['素材时长'].astype(str)
k7_material_desc_df_processed['素材时长'] = pd.to_timedelta(time_fixed, errors='coerce').dt.total_seconds()
# 素材类型
k7_material_desc_df_processed['素材创建时间'] = pd.to_datetime(k7_material_desc_df['素材创建时间'], errors='coerce')
# 素材来源
k7_material_desc_df_processed['素材来源'] = k7_material_desc_df['素材来源'].astype('category')
material_source_mapping = {cat: idx for idx, cat in enumerate(k7_material_desc_df_processed['素材来源'].cat.categories)}
k7_material_desc_df_processed['素材来源'] = k7_material_desc_df_processed['素材来源'].cat.codes # 将分类变量转换为数值编码
print(f"素材评估映射, 变量名: material_eval_mapping : {material_eval_mapping}")
print(f"素材来源映射, 变量名: material_source_mapping : {material_source_mapping}")
# Float 列
k7_materials_float_cols = ['整体展现次数','整体点击次数','整体消耗','基础消耗','整体千次展现费用']
for float_col in k7_materials_float_cols:
    if k7_material_desc_df[float_col].dtype == float:
        k7_material_desc_df_processed[float_col] = k7_material_desc_df[float_col]
    else:
        k7_material_desc_df_processed[float_col] = k7_material_desc_df[float_col].apply(lambda x: float(str(x).replace(',', '')) if isinstance(x, str) else np.nan)
# 百分比列
k7_materials_pct_cols = ['整体点击率','整体转化率']
for pct_col in k7_materials_pct_cols:
    k7_material_desc_df_processed[pct_col] = k7_material_desc_df[pct_col].apply(lambda x: float(str(x).replace('%', '')) if isinstance(x, str) else np.nan)

display(k7_material_desc_df_processed.head())
print("处理后的K7素材数据缺失值数量(ascending): ")
display(k7_material_desc_df_processed.replace("-", np.nan).isna().sum().sort_values(ascending=False).head(5).to_frame().T)

# 同样的，对素材名称做分析,判断是否是gibberish, 是否含有家庭相关的关键词、设备相关的关键词、唱歌相关的关键词、否定词、美女相关的关键词等
# 素材名称是否是meaningful的, 很多素材名称都是gibberish, 例如纯数字、日期、混剪、或者直接就是视频文件名等
gibberish_title_pattern = re.compile(r'^\d+$|\d+月\d+日|混剪|\.mp4$|\(\d+\)')
gibberish_title_filter = lambda x: bool(gibberish_title_pattern.search(x))
k7_material_desc_df_processed['meaningful_title'] = k7_material_desc_df_processed['素材名称'].apply(gibberish_title_filter) ^ 1

# 当日/近5日gibberish标题比例 (我觉得这个很影响观感....)
k7_material_desc_df_processed['datestamp'] = k7_material_desc_df_processed['素材创建时间'].dt.floor('D')
daily_gibberish_title_ratio_k7 = k7_material_desc_df_processed.groupby('datestamp')['meaningful_title'].apply(lambda x: 1 - x.mean()).reset_index(name='date_gibberish_title_ratio')
daily_gibberish_title_ratio_k7['rollong_5d_giibbersh_ratio'] = daily_gibberish_title_ratio_k7['date_gibberish_title_ratio'].rolling(window=5, min_periods=1).mean() # 计算当日gibberish标题比例的5日滚动平均
k7_material_desc_df_processed = pd.merge(k7_material_desc_df_processed, daily_gibberish_title_ratio_k7, on = 'datestamp', how = 'left')

# 含人的标题
family_title_keywords = ['爸爸', '妈妈', '儿子', '女儿', '兄弟', '姐妹', '爷爷', '奶奶', '外公', '外婆','老公','老婆','丈夫','妻子','姐','妹','哥','弟','父亲','母亲','儿女','兄妹','祖父','祖母','外祖父','外祖母','小伙子','家庭','家']
k7_material_desc_df_processed['family_kw_in_title'] = k7_material_desc_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in family_title_keywords))

# 设备标题
device_title_keywords = ['设施','设备','麦克风','音响','k歌','K歌','蓝牙','KTV']
k7_material_desc_df_processed['device_kw_in_title'] = k7_material_desc_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in device_title_keywords))

# 唱歌标题
singing_title_keywords = ['唱歌','唱了','唱了首','唱了个','唱了段','唱了首歌','唱了个歌','唱了段歌','唱了首曲','唱了个曲','唱了段曲','唱了首调','麦霸']
k7_material_desc_df_processed['singing_kw_in_title'] = k7_material_desc_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in singing_title_keywords))

# 否定词 (我发现有些标题里有否定词.....无语了)
negation_title_keywords = ['不要','千万别','不能','不行','别再','别看','别点','别点开','别点进去','别点了','别点那个','别点这个','别点那个了','别点这个了']
k7_material_desc_df_processed['negation_kw_in_title'] = k7_material_desc_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in negation_title_keywords))

# 美女, 网红...... 
beauty_title_keywords = ['美女', '网红', '女神', '小姐姐', '小仙女']
k7_material_desc_df_processed['beauty_kw_in_title'] = k7_material_desc_df_processed['素材名称'].apply(lambda x: any(keyword in str(x) for keyword in beauty_title_keywords))

k7_material_desc_df_processed['normalized_name'] =k7_material_desc_df_processed['素材名称'].apply(lambda x: hfc.normalize_name(x))

display(k7_material_desc_df_processed.head())

缺失值数量(ascending): 


,素材评估,素材来源,素材时长,素材ID,素材创建时间,素材名称,整体展现次数
0,990,151,1,1,1,0,0


素材ID为'-'的行: 


,素材名称,素材ID,素材评估,素材时长,素材创建时间,素材来源,整体展现次数,整体点击次数,整体点击率,整体转化率,整体消耗,基础消耗,整体点击单价,整体千次展现费用
1,AIGC动态创意素材集合,-,-,-,-,AIGC动态创意,"521,325","39,154",7.51%,0.91%,"98,400.03","98,400.03",2.51,188.75


只有一行, AIGC动态创意素材集合, 这一条并没有出现在上面的KINNO抖音每日绩效数据中, 可能是一个特殊的素材集合, 但由于没有对应的素材ID, 无法与绩效数据进行关联分析
去除AIGC这一行后 缺失值数量(ascending): 


...

素材评估映射, 变量名: material_eval_mapping : {'-': 0, '优质': 1, '优质，同质化': 2, '低效': 3, '可提升': 4, '同质化': 5, '同质化，可提升': 6, '首发': 7, '首发，优质': 8, '首发，低效': 9, '首发，同质化': 10, '首发，搬运': 11}
素材来源映射, 变量名: material_source_mapping : {'-': 0, '抖音号主页素材': 1, '本地上传': 2}


,素材ID,素材名称,素材评估,素材时长,素材创建时间,素材来源,整体展现次数,整体点击次数,整体消耗,基础消耗,整体千次展现费用,整体点击率,整体转化率
0,7582775222080798729,12月11日-K7-YL-(4)混剪,0,208.0,2026-01-12 22:31:50,2,2345141.0,85971.0,175636.77,175636.77,74.89,3.67,0.79
2,7600402383899377718,2月10日-K7-CM- 混剪-,4,225.0,2026-02-10 16:58:00,2,533821.0,23437.0,66624.88,66624.88,124.81,4.39,1.26
3,7601865320384036927,2月1日-K7-MY- 混剪- (2),5,244.0,2026-02-01 20:00:19,2,463324.0,17568.0,65190.14,65190.14,140.70,3.79,1.20
4,7605160196621090879,2月10日-K7-MY- (5),0,235.0,2026-02-10 17:05:58,2,419707.0,14026.0,44659.10,44659.10,106.41,3.34,1.01
5,7560103525498896393,10月11日-K7-MY- (4)混剪已发,0,200.0,2026-01-23 14:04:08,2,108128.0,5787.0,39952.23,39952.23,369.49,5.35,2.07


处理后的K7素材数据缺失值数量(ascending): 


,素材ID,素材名称,素材评估,素材时长,素材创建时间
0,0,0,0,0,0


,素材ID,素材名称,素材评估,素材时长,素材创建时间,素材来源,整体展现次数,整体点击次数,整体消耗,基础消耗,...,meaningful_title,datestamp,date_gibberish_title_ratio,rollong_5d_giibbersh_ratio,family_kw_in_title,device_kw_in_title,singing_kw_in_title,negation_kw_in_title,beauty_kw_in_title,normalized_name
0,7582775222080798729,12月11日-K7-YL-(4)混剪,0,208.0,2026-01-12 22:31:50,2,2345141.0,85971.0,175636.77,175636.77,...,False,2026-01-12,1.000000,0.200000,False,False,False,False,False,12月11日-K7-YL-(4)混剪
1,7600402383899377718,2月10日-K7-CM- 混剪-,4,225.0,2026-02-10 16:58:00,2,533821.0,23437.0,66624.88,66624.88,...,False,2026-02-10,0.932203,0.917113,False,False,False,False,False,2月10日-K7-CM-混剪
2,7601865320384036927,2月1日-K7-MY- 混剪- (2),5,244.0,2026-02-01 20:00:19,2,463324.0,17568.0,65190.14,65190.14,...,False,2026-02-01,0.935484,0.912473,False,False,False,False,False,2月1日-K7-MY-混剪-(2)
3,7605160196621090879,2月10日-K7-MY- (5),0,235.0,2026-02-10 17:05:58,2,419707.0,14026.0,44659.10,44659.10,...,False,2026-02-10,0.932203,0.917113,False,False,False,False,False,2月10日-K7-MY-(5)
4,7560103525498896393,10月11日-K7-MY- (4)混剪已发,0,200.0,2026-01-23 14:04:08,2,108128.0,5787.0,39952.23,39952.23,...,False,2026-01-23,1.000000,0.786667,False,False,False,False,False,10月11日-K7-MY-(4)混剪已发


In [ ]:
# 同样, 对于素材的标题进行LLM评估, 看看素材标题的marketing_score, attention_score, promotion_score分别是多少
MAX_WORKERS = 16 # 并发线程数
k7_meaningful_titles = k7_material_desc_df_processed[k7_material_desc_df_processed['meaningful_title'] == True]['素材名称'].unique()

# print(f"K7 DataFrame 有 {len(k7_meaningful_titles)} 个不同的有意义的标题")
# # 执行多线程评估
# print(f"开始评估 K7 素材的 {len(k7_meaningful_titles)} 个标题，使用 {MAX_WORKERS} 个线程...")
# start_time = time.time()    
# k7_results = hfc.evaluate_titles_multithread(
#     titles=k7_meaningful_titles,
#     api_key= os.getenv("SILICONFLOW_API"),
#     prompt_template = PromptTemplate.TITLE_EVAL_PROMPT,
#     max_workers=16  # 可根据需要调整
# )
# elapsed_time = time.time() - start_time
# k7_meaningful_titles_analysis_result_df = pd.DataFrame(k7_results)
# # 数值转换
# if not k7_meaningful_titles_analysis_result_df.empty:
#     # 转换分数列为数值类型
#     score_cols = ['marketing_score', 'attention_score', 'promotion_score']
#     for col in score_cols:
#         if col in k7_meaningful_titles_analysis_result_df.columns:
#             k7_meaningful_titles_analysis_result_df[col] = pd.to_numeric(k7_meaningful_titles_analysis_result_df[col], errors='coerce')
# # 输出统计信息
# print(f"\n完成, 耗时: {elapsed_time:.2f} 秒")
# print(f"成功: {k7_meaningful_titles_analysis_result_df['status'].value_counts().get('success', 0)} 条")
# print(f"失败: {k7_meaningful_titles_analysis_result_df['status'].value_counts().get('failed', 0)} 条")
# k7_meaningful_titles_analysis_result_df['source'] = 'k7_material_desc'

# k7_meaningful_titles_embedding_results = hfc.embed_titles_multithread(
#     titles=k7_meaningful_titles,
#     api_key=os.getenv("SILICONFLOW_API"),
#     model="BAAI/bge-large-zh-v1.5",
#     max_workers=8,
#     timeout=40,
#     max_retries=3,
# )
# k7_meaningful_titles_embedding_df = pd.DataFrame(k7_meaningful_titles_embedding_results)
# k7_meaningful_titles_analysis_result_df = pd.merge(k7_meaningful_titles_analysis_result_df, k7_meaningful_titles_embedding_df, on = 'meaningful_title', how='left',suffixes=('_eval', '_embed'))

# # 将结果保存到Excel文件
# k7_meaningful_titles_analysis_result_df.to_excel(os.path.join(Processed_DATA_DIR, "k7_meaningful_titles_analysis_result.xlsx"), index=False)

# 结果读取和展示
k7_meaningful_titles_analysis_result_df = pd.read_excel(os.path.join(Processed_DATA_DIR, "k7_meaningful_titles_analysis_result.xlsx")) # 读取评估结果
# 显示结果
print("\nK7 素材标题评估结果预览:")
display(k7_meaningful_titles_analysis_result_df.head())

K7 DataFrame 有 98 个不同的有意义的标题
开始评估 K7 素材的 98 个标题，使用 16 个线程...


评估标题进度: 100%|██████████| 98/98 [02:37<00:00,  1.61s/个, 成功=98, 失败=0]



完成, 耗时: 157.82 秒
成功: 98 条
失败: 0 条


Embedding进度: 100%|██████████| 98/98 [00:02<00:00, 36.81个/s, 成功=98, 失败=0]



K7 素材标题评估结果预览:


,marketing_score,attention_score,promotion_score,keywords,overall_comment,meaningful_title,status_eval,error_eval,source,status_embed,error_embed,embedding,embedding_dim,model
0,4,4,4,"['电视机', '闲置', '音响', '智能音箱', '家庭KTV']",直击闲置痛点，场景化推荐，标签清晰，引导性强。,如果你家电视机一直闲置着不看，我建议你用上这个东西#音响 #智能音箱 #家庭KTV,success,NaN,k7_material_desc,success,NaN,"[-0.022035364, -0.021482406, 0.0055330326, -0....",1024,BAAI/bge-large-zh-v1.5
1,4,4,4,"['KTV', '装', '多少钱', '家庭KTV']",直击用户痛点，价格敏感度高，利于转化。,家里能不能装KTV，装一套KTV到底需要多少钱？#音响 #智能音箱 #家庭KTV,success,NaN,k7_material_desc,success,NaN,"[-0.03751251, -0.0132604055, -0.008807918, -0....",1024,BAAI/bge-large-zh-v1.5
2,4,5,4,"['KTV', '搬回家', '音响', '家庭KTV', '智能音响']",制造反差吸引眼球，场景化营销明确，标签丰富。,想不到现在的人不去KTV而是把KTV搬回家#音响 #家庭KTV #智能音响 #生活技巧,success,NaN,k7_material_desc,success,NaN,"[-0.023723664, -0.031004205, -0.02085107, -0.0...",1024,BAAI/bge-large-zh-v1.5
3,4,4,4,"['听歌', '唱歌', '功放机', '智能音箱', '家庭KTV']",痛点明确，反向营销吸引人，标签精准。,喜欢听歌唱歌朋友可不要再去装功放机了 #音响 #智能音箱 #家庭KTV,success,NaN,k7_material_desc,success,NaN,"[-0.015712466, -0.028223792, -0.02817492, 0.01...",1024,BAAI/bge-large-zh-v1.5
4,5,4,5,"['音响', '家庭KTV', '金运K7']",直击痛点，场景感强，品牌露出明显，利于转化。,唱歌不需要太复杂！一台音响！在家就能打造家庭KTV！ #家庭KTV#金运K7#金运家庭KTV,success,NaN,k7_material_desc,success,NaN,"[-0.011177967, -0.03283488, -0.0011295006, 0.0...",1024,BAAI/bge-large-zh-v1.5


# 4. 匹配视频(并不是全有)

In [19]:
# 匹配视频文件

# 1) 读取 K7 两个文件夹中的视频文件名
popular_mv_files = os.listdir(K7_GOOD_PERF_DATA_FOLDER)
ordinary_mv_files = os.listdir(K7_BAD_PERF_DATA_FOLDER)
print(f"K7绩效表现好的视频文件数量: {len(popular_mv_files)}")
print(f"K7绩效表现差的视频文件数量: {len(ordinary_mv_files)}")

# 2) 构建“标准化文件名 -> 原始文件信息列表”的字典
#    - 使用 hfc.normalize_name 统一命名，减少空格/连字符差异导致的匹配失败
#    - value 用 list 是为了防止重名文件被覆盖
#    - 元组第二位布尔值表示是否来自爆款目录(True=爆款, False=普通)
mv_name_dict = defaultdict(list)
for f in popular_mv_files:
    mv_name_dict[hfc.normalize_name(f)].append((f, True))
for f in ordinary_mv_files:
    mv_name_dict[hfc.normalize_name(f)].append((f, False))

# 3) 在金运抖音每日绩效表中，用 normalized_title 去匹配视频文件
print(f"kinno_dy_daily_perf_df_processed 中一共有 {kinno_dy_daily_perf_df_processed['normalized_title'].nunique()} 个不同的normalized_title")
kinno_dy_daily_perf_df_processed['is_popular_mv'] = kinno_dy_daily_perf_df_processed['normalized_title'].apply(
    lambda x: mv_name_dict.get(x, np.nan)
    )
print("在金运抖音每日绩效数据中, 根据视频文件名匹配到的视频数量: ", kinno_dy_daily_perf_df_processed['is_popular_mv'].dropna().shape[0])

# 4) 在 K7 素材表中，用 normalized_name 去匹配视频文件
print(f"\nk7_material_desc_df_processed 中一共有 {k7_material_desc_df_processed['normalized_name'].nunique()} 个不同的normalized_name")
k7_material_desc_df_processed['is_popular_mv'] = k7_material_desc_df_processed['normalized_name'].apply(
    lambda x: mv_name_dict.get(x, np.nan)
    )
print("在K7素材数据中, 根据视频文件名匹配到的视频数量: ", k7_material_desc_df_processed['is_popular_mv'].dropna().shape[0])

# 5) 展示匹配成功的 K7 素材，便于人工检查
print("\nK7素材数据中匹配到的视频: ")
display(k7_material_desc_df_processed[k7_material_desc_df_processed['is_popular_mv'].notna()])

# 6) 保存匹配结果到Excel
k7_material_desc_df_processed.to_excel(os.path.join(Processed_DATA_DIR, "k7_material_desc_with_mv_match.xlsx"), index=False)
kinno_dy_daily_perf_df_processed.to_excel(os.path.join(Processed_DATA_DIR, "dy_daily_perf_with_mv_match.xlsx"), index=False)

K7绩效表现好的视频文件数量: 246
K7绩效表现差的视频文件数量: 100
kinno_dy_daily_perf_df_processed 中一共有 1862 个不同的normalized_title
在金运抖音每日绩效数据中, 根据视频文件名匹配到的视频数量:  0

k7_material_desc_df_processed 中一共有 1244 个不同的normalized_name
在K7素材数据中, 根据视频文件名匹配到的视频数量:  67

K7素材数据中匹配到的视频: 


,素材ID,素材名称,素材评估,素材时长,素材创建时间,素材来源,整体展现次数,整体点击次数,整体消耗,基础消耗,...,datestamp,date_gibberish_title_ratio,rollong_5d_giibbersh_ratio,family_kw_in_title,device_kw_in_title,singing_kw_in_title,negation_kw_in_title,beauty_kw_in_title,normalized_name,is_popular_mv
68,7601865319074398249,2月1日-K7-MY-- (12)已发,8,70.0,2026-02-01 20:00:19,2,53872.0,2586.0,5133.49,5133.49,...,2026-02-01,0.935484,0.912473,False,False,False,False,False,2月1日-K7-MY-(12)已发,"[(2月1日-K7-MY-- (12)已发.mp4, True)]"
255,7611439619520741417,2月26日-k7-XT (8),1,44.0,2026-02-27 15:13:23,2,4091.0,425.0,624.18,624.18,...,2026-02-27,0.977273,0.899392,False,False,False,False,False,2月26日-k7-XT-(8),"[(2月26日-k7-XT (8).mp4, False)]"
323,7614694447704293391,3月7日-K7-YL- (3)已发,0,70.0,2026-03-08 10:05:47,2,5714.0,207.0,367.55,367.55,...,2026-03-08,0.920000,0.917114,False,False,False,False,False,3月7日-K7-YL-(3)已发,"[(3月7日-K7-YL- (3)已发.mp4, True)]"
344,7611439606803447849,2月26日-k7-XT (5),0,55.0,2026-02-27 15:13:23,2,1326.0,164.0,308.09,308.09,...,2026-02-27,0.977273,0.899392,False,False,False,False,False,2月26日-k7-XT-(5),"[(2月26日-k7-XT (5).mp4, True), (2月26日-k7-XT (..."
369,7615223367035338803,3月9日-K7-MY- (7)已发,7,34.0,2026-03-09 19:56:33,2,3916.0,502.0,245.54,245.54,...,2026-03-09,0.937500,0.917114,False,False,False,False,False,3月9日-K7-MY-(7)已发,"[(3月9日-K7-MY- (7)已发.mp4, False)]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1014,7621749618532139058,3月26日-K7-MY- 桌搭- (9)已发,7,24.0,2026-03-29 18:52:45,2,60.0,5.0,4.26,4.26,...,2026-03-29,0.859375,0.905208,False,False,False,False,False,3月26日-K7-MY-桌搭-(9)已发,"[(3月26日-K7-MY- 桌搭- (9)已发.mp4, False)]"
1102,7611439615297224758,2月26日-k7-XT (10),0,63.0,2026-02-27 15:13:23,2,15.0,3.0,1.35,1.35,...,2026-02-27,0.977273,0.899392,False,False,False,False,False,2月26日-k7-XT-(10),"[(2月26日-k7-XT (10).mp4, True), (2月26日-k7-XT ..."
1138,7611439599355297828,2月26日-k7-XT (3),0,65.0,2026-02-27 15:13:23,2,5.0,0.0,0.73,0.73,...,2026-02-27,0.977273,0.899392,False,False,False,False,False,2月26日-k7-XT-(3),"[(2月26日-k7-XT (3).mp4, True)]"
1156,7603726652846948362,2月6日-K7-MY- (2)已发,0,61.0,2026-02-06 20:23:16,2,6.0,0.0,0.53,0.53,...,2026-02-06,0.956522,0.962586,False,False,False,False,False,2月6日-K7-MY-(2)已发,"[(2月6日-K7-MY- (2)已发.mp4, True)]"
